# K-means Clustering: Snack Map Simulator

K-means groups nearby data points into `k` clusters.

The method grew from signal processing and statistics, where researchers needed simple ways to summarize clouds of points. Today k-means appears in image compression, customer segmentation, document grouping, sensor analysis, and exploratory data science.

In this notebook, you will build it with small objects: snack spots, cluster centers, assignments, and a runner that moves centers until groups settle.

<details>
<summary>Big idea</summary>

Each round has two moves: assign every point to its nearest center, then move each center to the average location of its assigned points.

</details>

## 1. The Mental Model

K-means is a clustering algorithm:

- **Point**: one item with coordinates
- **Cluster**: a group of nearby points
- **Center**: the current middle of a cluster
- **Assignment**: choosing the nearest center for a point
- **Round**: assign points, then move centers

K-means repeats until the centers barely move or you hit the round limit.

<details>
<summary>Hint: what does `k` mean?</summary>

`k` is the number of clusters you ask the algorithm to find. If `k = 3`, the algorithm keeps three centers.

</details>

## 2. Build the Objects

Implementation plan:

1. `SnackSpot` stores a point on the map.
2. `Center` stores the current location of a cluster center.
3. `ClusterStep` records each simulation round.
4. `KMeansRunner` owns assignment, movement, and stopping.

<details>
<summary>Implementation hint</summary>

Use squared distance to compare centers. It avoids square roots and gives the same nearest-center answer.

</details>

In [ ]:
from dataclasses import dataclass


### Define how to store a point on the map

- 

In [ ]:
@dataclass(frozen=True)
class SnackSpot:
    name: str
    east: float
    north: float

    def squared_distance_to(self, center: "Center") -> float:
        east_gap = self.east - center.east
        north_gap = self.north - center.north
        return east_gap * east_gap + north_gap * north_gap

    def __str__(self) -> str:
        return f"{self.name}({self.east}, {self.north})"

### Define how to store cluster center

-


In [ ]:
@dataclass(frozen=True)
class Center:
    label: str
    east: float
    north: float

    def moved_distance_to(self, other: "Center") -> float:
        east_gap = self.east - other.east
        north_gap = self.north - other.north
        return (east_gap * east_gap + north_gap * north_gap) ** 0.5

    def __str__(self) -> str:
        return f"{self.label}=({self.east:.2f}, {self.north:.2f})"


### Define Cluster Step

- 

In [ ]:
@dataclass
class ClusterStep:
    round_number: int
    centers: list[Center]
    assignments: dict[str, list[SnackSpot]]
    moved_distances: dict[str, float]
    note: str


### Define Cluster Result

- 

In [ ]:
@dataclass
class ClusterResult:
    centers: list[Center]
    assignments: dict[str, list[SnackSpot]]
    steps: list[ClusterStep]


### Define the K-Means Algorithm

- 

In [ ]:
class KMeansRunner:
    def __init__(self, spots: list[SnackSpot], starting_centers: list[Center]):
        if not spots:
            raise ValueError("K-means needs at least one point.")
        if not starting_centers:
            raise ValueError("K-means needs at least one center.")
        self.spots = spots
        self.starting_centers = starting_centers

    def run(self, rounds: int = 8, tolerance: float = 0.01) -> ClusterResult:
        centers = self.starting_centers.copy()
        steps: list[ClusterStep] = []

        for round_number in range(1, rounds + 1):
            assignments = self._assign(centers)
            new_centers = self._move_centers(centers, assignments)
            moved_distances = {
                old_center.label: old_center.moved_distance_to(new_center)
                for old_center, new_center in zip(centers, new_centers)
            }
            centers = new_centers
            steps.append(
                ClusterStep(
                    round_number=round_number,
                    centers=centers.copy(),
                    assignments=assignments,
                    moved_distances=moved_distances,
                    note=f"Round {round_number}: assign spots, then move centers.",
                )
            )

            if max(moved_distances.values()) <= tolerance:
                break

        final_assignments = self._assign(centers)
        return ClusterResult(centers=centers, assignments=final_assignments, steps=steps)

    def _assign(self, centers: list[Center]) -> dict[str, list[SnackSpot]]:
        assignments = {center.label: [] for center in centers}

        for spot in self.spots:
            closest_center = min(centers, key=lambda center: spot.squared_distance_to(center))
            assignments[closest_center.label].append(spot)

        return assignments

    def _move_centers(self, centers: list[Center], assignments: dict[str, list[SnackSpot]]) -> list[Center]:
        new_centers = []

        for center in centers:
            assigned_spots = assignments[center.label]
            if not assigned_spots:
                new_centers.append(center)
                continue

            average_east = sum(spot.east for spot in assigned_spots) / len(assigned_spots)
            average_north = sum(spot.north for spot in assigned_spots) / len(assigned_spots)
            new_centers.append(Center(center.label, average_east, average_north))

        return new_centers

## 3. Create a Snack Map

Now make a small set of snack spots on a simple coordinate map.

The starting centers are guesses. K-means will move them toward the middle of nearby spots.

<details>
<summary>Hint: why starting centers matter</summary>

K-means can land on different clusters depending on where centers start. That is why real projects often run it several times.

</details>

In [2]:
spots = [
    SnackSpot("Taco Stand", 1.0, 1.0),
    SnackSpot("Donut Cart", 2.0, 1.5),
    SnackSpot("Coffee Bar", 1.5, 2.5),
    SnackSpot("Sushi Stop", 8.0, 8.0),
    SnackSpot("Ramen Van", 7.0, 7.5),
    SnackSpot("Tea Hut", 8.5, 6.8),
    SnackSpot("Pizza Booth", 1.0, 8.0),
    SnackSpot("Salad Shop", 2.0, 7.5),
    SnackSpot("Smoothie Spot", 3.0, 8.5),
]

starting_centers = [
    Center("A", 0.0, 0.0),
    Center("B", 9.0, 9.0),
    Center("C", 0.0, 9.0),
]

print("Snack spots:")
for spot in spots:
    print(" ", spot)

print("\nStarting centers:")
for center in starting_centers:
    print(" ", center)

Snack spots:
  Taco Stand(1.0, 1.0)
  Donut Cart(2.0, 1.5)
  Coffee Bar(1.5, 2.5)
  Sushi Stop(8.0, 8.0)
  Ramen Van(7.0, 7.5)
  Tea Hut(8.5, 6.8)
  Pizza Booth(1.0, 8.0)
  Salad Shop(2.0, 7.5)
  Smoothie Spot(3.0, 8.5)

Starting centers:
  A=(0.00, 0.00)
  B=(9.00, 9.00)
  C=(0.00, 9.00)


## 4. Run K-means

The runner returns a `ClusterResult` with three things:

- `centers`: final center locations
- `assignments`: which spots belong to each center
- `steps`: snapshots for replaying the rounds

<details>
<summary>Quick check</summary>

The lower-left snack spots should end up together, the upper-right spots should end up together, and the upper-left spots should end up together.

</details>

In [3]:
runner = KMeansRunner(spots, starting_centers)
result = runner.run(rounds=8)

print("Final centers:")
for center in result.centers:
    print(" ", center)

print("\nFinal clusters:")
for label, assigned_spots in result.assignments.items():
    names = ", ".join(spot.name for spot in assigned_spots)
    print(f"{label}: {names}")

Final centers:
  A=(1.50, 1.67)
  B=(7.83, 7.43)
  C=(2.00, 8.00)

Final clusters:
A: Taco Stand, Donut Cart, Coffee Bar
B: Sushi Stop, Ramen Van, Tea Hut
C: Pizza Booth, Salad Shop, Smoothie Spot


## 5. Replay the Rounds

The replay shows how centers move and which spots belong to each cluster.

<details>
<summary>Hint: what means converged?</summary>

Converged means the centers barely move between rounds, so another round would not change much.

</details>

**Trace model.** Define `ClusterReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class ClusterReplay:
    def __init__(self, steps: list[ClusterStep]):
        self.steps = steps

    def show(self, limit: int | None = None) -> None:
        selected_steps = self.steps if limit is None else self.steps[:limit]

        for step in selected_steps:
            print(step.note)
            print("  centers:", self._format_centers(step.centers))
            print("  moved  :", self._format_movement(step.moved_distances))
            for label, assigned_spots in step.assignments.items():
                print(f"  {label}: {self._format_spots(assigned_spots)}")
            print()

    def _format_centers(self, centers: list[Center]) -> str:
        return ", ".join(str(center) for center in centers)

    def _format_movement(self, moved_distances: dict[str, float]) -> str:
        return ", ".join(f"{label}={distance:.2f}" for label, distance in moved_distances.items())

    def _format_spots(self, assigned_spots: list[SnackSpot]) -> str:
        return ", ".join(spot.name for spot in assigned_spots) or "empty"


**Example state.** Create `replay`, the concrete values used in the next run.


In [ ]:
replay = ClusterReplay(result.steps)

replay.show()


## 6. Your Experiments

Try changing one thing at a time:

- Move a snack spot closer to another group
- Add a new snack spot
- Change the starting centers
- Try only two centers instead of three

<details>
<summary>Challenge</summary>

Predict whether the final clusters will change before you run the cell. Then compare your guess with the output.

</details>

In [6]:
experiment_centers = [
    Center("A", 1.0, 1.0),
    Center("B", 8.0, 8.0),
]

experiment_runner = KMeansRunner(spots, experiment_centers)
experiment_result = experiment_runner.run(rounds=8)

print("Two-center experiment:")
for center in experiment_result.centers:
    print(" ", center)

print("\nClusters:")
for label, assigned_spots in experiment_result.assignments.items():
    names = ", ".join(spot.name for spot in assigned_spots)
    print(f"{label}: {names}")

Two-center experiment:
  A=(1.50, 1.67)
  B=(4.92, 7.72)

Clusters:
A: Taco Stand, Donut Cart, Coffee Bar
B: Sushi Stop, Ramen Van, Tea Hut, Pizza Booth, Salad Shop, Smoothie Spot


## Visual Trace + Rigor Studio

**Problem frame.** Partition points by alternating nearest centers and center updates.

**Interactive animation target.** Animate assignments, centroid movement, and objective value per iteration.

**Correctness handle.** Each assignment/update phase does not increase within-cluster squared error.

**Complexity handle.** O(iterations * k * n * dimensions).

**Failure mode to test.** Different initial centers can lead to different local optima.

**Studio task.** Run three initializations and compare the final clusters and objective values.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, kmeans_trace, render_kmeans, render_trace_table

points = [(1, 1), (1.2, 1.4), (0.8, 0.7), (4, 4), (4.3, 3.8), (3.7, 4.2)]
centers = [(0.5, 1.8), (4.8, 3.5)]
trace = kmeans_trace(points, centers, steps=4)
display(render_trace_table(trace))
AlgorithmPlayer(trace, render_kmeans).display()
